# 02 — Replicate the 2022 phenotype simulations**What this does.** Reproduces the Hope College team's Summer-2022 growth/no-growthpredictions using their own method, run with current code.**The method is not inferred — it is theirs.** It comes from Clayton Piehl's ownnotebook (`notebooks/data/BioLogMacTest.ipynb`):```pythongmm      = get_media("Carbon-Pyruvic-Acid", "KBaseMedia")template = get_template("GramNegModelTemplateV4", "NewKBaseModelTemplates")biolog   = get_phenotypeset("ecoli_biolog", 93541)      # 324 phenotypesfor model in old_models:                                # ws 119455    model.objective = "bio1"    gf  = MSGapfill(model, [template], [], {}, {}, [])    sol = gf.run_gapfilling(gmm, "bio1")    gf.integrate_gapfill_solution(sol)    simulate(gf.mdlutl, biolog, add_missing_exchanges=True)```They **gapfilled**; they did not supplement the medium. The saved 2022 models areauxotrophic — on a bare Biolog medium every one of them returns zero biomass,glucose included — and gapfilling on pyruvate is what makes them viable.**What we score against.** `notebooks/data/BiologMacTestOutput_On-Off.csv`, Clayton'sown 0/1 growth calls. We verified these are the exact binarisation of the mastersheet:**0 disagreements across 168,156 cells**.**And separately, against experiment.** `ecoli_biolog` carries real measured Biologvalues. Agreeing with the 2022 run only proves we reproduced a pipeline; agreeing withthe experiment is the only thing that says a prediction is *right*.

## Step 1 — Load the reference dataClayton's 2022 calls (the replication target) and the experimental Biolog values (theaccuracy target). The Biolog media come back with **CO2 uptake blocked** — see Step 3for why that matters enormously.

In [ ]:
%run util.pyreference = load_reference_calls()                 # Clayton's 2022 0/1 callsexperiment = get_experimental_calls()              # real measured Biolog growthmedia = get_biolog_media()                         # 324 phenotypes, CO2 uptake BLOCKEDprint(f"reference calls : {reference.shape[0]} media x {reference.shape[1]} models")print(f"experimental    : {len(experiment)} media with a measured value")print(f"biolog media    : {len(media)} phenotypes")session.cache.save("reference_calls", reference)session.cache.save("experimental_calls", experiment)

## Step 2 — Partition the models: old vs new**By object NAME, not by workspace.** The narrative workspace (119455) contains *both*generations — 519 `<gid>.fbamodel` from 2022 and 524 `genomeset__*` from 2025 — so aworkspace-based split silently labels the 2025 rebuilds as "old" and contaminates thereference arm. This was a real bug here; see `partition_models`.

In [ ]:
%run util.pyref_genomes, media_names, watershed_df = load_watershed_sheet()partitions, report = partition_models(ref_genomes)print(f"old models: {len(partitions['old'])}/{len(ref_genomes)} genomes  (<gid>.fbamodel, 2022)")print(f"new models: {len(partitions['new'])}/{len(ref_genomes)} genomes  (genomeset__*, 2025)")session.cache.save("model_partitions", partitions)

## Step 3 — Replicate one genome, and check the model is not broken`gapfill_model()` runs two guards after gapfilling, and they are **not** the same check:| guard | meaning | severity ||---|---|---|| **free lunch** | biomass from **minerals alone** — no carbon of any kind | **FATAL.** Thermodynamically impossible; no media-level fix exists (nothing to block); the model "grows" on all 324 media regardless of contents. Its predictions are meaningless. || **autotrophic** | biomass from **CO2 + minerals** | Warn. Curable — block CO2 uptake (which the media already specify via `maxFlux=0`) and the route never fires. Predictions stay sound. |Both had to be found the hard way. Gapfilling adds mass-imbalanced reactions that closea futile cycle, manufacturing the reducing power to fix carbon; and`MSGrowthPhenotype.build_media()` discards the media's own `maxFlux=0` on CO2, handingthe model carbon the medium forbids. Together they make *E. coli* autotrophic, and theonly symptom is that the model quietly "grows" on substrates it cannot eat.Testing the two together — minerals **and** CO2 in one shot — cannot tell them apart,and reports an *incurable* model as mitigated. Test them separately. Always.

In [ ]:
%run util.pypartitions = session.cache.load("model_partitions")media = get_biolog_media()reference = session.cache.load("reference_calls")GENOME = "562.55380"model = get_msfba().get_model(_latest_ref(partitions["old"][GENOME]))mdlutl, info = gapfill_model(model)print(f"{GENOME}: gapfill added {info['n_reactions_added']} reactions")print(f"   free lunch (minerals alone) : {info['mineral_biomass']:.4f}  "      f"{'<<< BROKEN' if info['free_lunch'] else 'ok'}")print(f"   autotrophic (CO2+minerals)  : {info['co2_biomass']:.4f}  "      f"{'<<< carries a carbon-fixation route (CO2 is blocked, so it never fires)' if info['autotrophic'] else 'ok'}")rows = simulate_biolog_panel(mdlutl, media)vs_2022 = score_calls(rows, reference[f"{GENOME}.fbamodel"])vs_exp  = score_calls(rows, experiment := session.cache.load("experimental_calls"))print(f"\nvs Clayton's 2022 calls : {vs_2022['accuracy']:.3f}  "      f"(CP={vs_2022['CP']} CN={vs_2022['CN']} FP={vs_2022['FP']} FN={vs_2022['FN']})")print(f"vs EXPERIMENTAL Biolog  : {vs_exp['accuracy']:.3f}  "      f"(CP={vs_exp['CP']} CN={vs_exp['CN']} FP={vs_exp['FP']} FN={vs_exp['FN']})")

## Step 4 — The full sweep519 genomes x 3 model arms is ~1,500 gapfills, far too much for a notebook cell. It runsas a checkpointed, resumable script instead:```bashpython scripts/three_arm_sweep.py --output NBOutput/three_arm_sweep.tsv --workers 6```Re-running skips pairs already present, so an interrupted sweep simply continues. Theanalysis of its output is notebook 03.

In [ ]:
%run util.pysweep_path = NBOUTPUT_DIR / "three_arm_sweep.tsv"if sweep_path.exists():    sweep = pd.read_csv(sweep_path, sep="\t")    done = len(sweep)    print(f"{done} of an expected {519*3} rows ({done/(519*3):.0%})")    print(sweep["status"].value_counts().to_string())else:    print(f"{sweep_path} not found -- run scripts/three_arm_sweep.py first")